# Reinforcement Learning for Games
## Tic-Tac-Toe AI using Q-Learning
**Author: Shaik Rehaman**

In [ ]:
# ============================================================
# CODETECH INTERNSHIP - TASK 3
# REINFORCEMENT LEARNING FOR GAMES
# Tic-Tac-Toe AI using Q-Learning
# Author: Shaik Rehaman
# ============================================================

import os
import random
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict

# 1. SETUP
random.seed(42)
np.random.seed(42)

os.makedirs("models", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

# 2. TIC-TAC-TOE ENVIRONMENT

WIN_LINES = [
    (0, 1, 2), (3, 4, 5), (6, 7, 8),
    (0, 3, 6), (1, 4, 7), (2, 5, 8),
    (0, 4, 8), (2, 4, 6)
]

def winner(board):
    for a, b, c in WIN_LINES:
        total = board[a] + board[b] + board[c]
        if total == 3:
            return 1
        if total == -3:
            return -1
    return 0

def legal_actions(board):
    return [i for i in range(9) if board[i] == 0]

def is_draw(board):
    return all(x != 0 for x in board) and winner(board) == 0

def get_state(board):
    return tuple(board)

def display_board(board):
    symbols = {1: "X", -1: "O", 0: " "}
    for i in range(3):
        print(" | ".join(symbols[board[i*3+j]] for j in range(3)))
        if i < 2:
            print("---------")

# 3. Q-LEARNING AGENT

class QLearningAgent:

    def __init__(self):
        self.q_table = defaultdict(lambda: np.zeros(9))
        self.alpha = 0.15
        self.gamma = 0.95
        self.epsilon = 1.0
        self.epsilon_min = 0.05
        self.epsilon_decay = 0.9997

    def choose_action(self, board, training=True):

        actions = legal_actions(board)

        if training and random.random() < self.epsilon:
            return random.choice(actions)

        state = get_state(board)
        values = self.q_table[state]

        best_value = max(values[a] for a in actions)
        best_actions = [
            a for a in actions if values[a] == best_value
        ]

        return random.choice(best_actions)

    def update(self, state, action, reward, next_state, done):

        old_value = self.q_table[state][action]

        if done:
            target = reward
        else:
            next_actions = legal_actions(list(next_state))

            if next_actions:
                future_value = max(
                    self.q_table[next_state][a]
                    for a in next_actions
                )
            else:
                future_value = 0

            target = reward + self.gamma * future_value

        self.q_table[state][action] += (
            self.alpha * (target - old_value)
        )

    def decay_epsilon(self):
        self.epsilon = max(
            self.epsilon_min,
            self.epsilon * self.epsilon_decay
        )

# 4. TRAINING EPISODE

def train_episode(agent):

    board = [0] * 9
    trajectory = []
    final_reward = 0

    while True:

        # AI move (X)
        state = get_state(board)
        action = agent.choose_action(board, training=True)
        board[action] = 1

        if winner(board) == 1:
            trajectory.append(
                (state, action, 1, get_state(board), True)
            )
            final_reward = 1
            break

        if is_draw(board):
            trajectory.append(
                (state, action, 0, get_state(board), True)
            )
            break

        # Random opponent move (O)
        opponent_action = random.choice(legal_actions(board))
        board[opponent_action] = -1

        if winner(board) == -1:
            trajectory.append(
                (state, action, -1, get_state(board), True)
            )
            final_reward = -1
            break

        if is_draw(board):
            trajectory.append(
                (state, action, 0, get_state(board), True)
            )
            break

        next_state = get_state(board)

        trajectory.append(
            (state, action, 0, next_state, False)
        )

    # Update Q-values from the episode transitions
    for state, action, reward, next_state, done in reversed(trajectory):
        agent.update(state, action, reward, next_state, done)

    agent.decay_epsilon()

    return final_reward

# 5. TRAIN THE AI

EPISODES = 30000

agent = QLearningAgent()
training_results = []

print("Training started...")

for episode in range(1, EPISODES + 1):

    result = train_episode(agent)
    training_results.append(result)

    if episode % 5000 == 0:

        recent = training_results[-5000:]

        win_rate = recent.count(1) / len(recent)

        print(
            f"Episode: {episode}/{EPISODES} | "
            f"Win Rate: {win_rate:.2%} | "
            f"Epsilon: {agent.epsilon:.3f}"
        )

print("Training completed!")

# 6. EVALUATION

def evaluate_game(agent):

    board = [0] * 9

    while True:

        # AI move
        action = agent.choose_action(board, training=False)
        board[action] = 1

        if winner(board) == 1:
            return "Win"

        if is_draw(board):
            return "Draw"

        # Random opponent move
        opponent_action = random.choice(legal_actions(board))
        board[opponent_action] = -1

        if winner(board) == -1:
            return "Loss"

        if is_draw(board):
            return "Draw"


EVALUATION_GAMES = 3000

evaluation_results = []

print("\nEvaluating trained AI...")

for _ in range(EVALUATION_GAMES):
    evaluation_results.append(evaluate_game(agent))

wins = evaluation_results.count("Win")
draws = evaluation_results.count("Draw")
losses = evaluation_results.count("Loss")

win_rate = wins / EVALUATION_GAMES
draw_rate = draws / EVALUATION_GAMES
loss_rate = losses / EVALUATION_GAMES

print("\nEVALUATION RESULTS")
print("------------------")
print("Total Games:", EVALUATION_GAMES)
print("Wins:", wins)
print("Draws:", draws)
print("Losses:", losses)
print("Win Rate:", round(win_rate * 100, 2), "%")
print("Draw Rate:", round(draw_rate * 100, 2), "%")
print("Loss Rate:", round(loss_rate * 100, 2), "%")

# 7. SAVE EVALUATION RESULTS

metrics = pd.DataFrame({
    "Metric": ["Total Games", "Wins", "Draws", "Losses"],
    "Value": [EVALUATION_GAMES, wins, draws, losses]
})

metrics.to_csv(
    "outputs/evaluation_metrics.csv",
    index=False
)

history = pd.DataFrame({
    "Episode": range(1, EPISODES + 1),
    "Result": training_results
})

history.to_csv(
    "outputs/training_history.csv",
    index=False
)

# 8. TRAINING GRAPH

window = 1000

win_rates = [
    np.mean(
        np.array(training_results[i:i+window]) == 1
    )
    for i in range(0, EPISODES, window)
]

plt.figure(figsize=(10, 5))

plt.plot(
    range(1, len(win_rates) + 1),
    win_rates,
    marker="o"
)

plt.title("Q-Learning AI Training Performance")
plt.xlabel("Training Blocks (1,000 Episodes Each)")
plt.ylabel("Win Rate")
plt.ylim(0, 1)
plt.grid(True)

plt.savefig(
    "outputs/training_performance.png",
    dpi=150
)

plt.show()

# 9. EVALUATION GRAPH

plt.figure(figsize=(7, 5))

plt.bar(
    ["Wins", "Draws", "Losses"],
    [wins, draws, losses]
)

plt.title("AI Evaluation Results")
plt.ylabel("Number of Games")

plt.savefig(
    "outputs/evaluation_results.png",
    dpi=150
)

plt.show()

# 10. SAVE TRAINED MODEL

joblib.dump(
    dict(agent.q_table),
    "models/q_learning_model.pkl"
)

# 11. SAVE TEXT REPORT

with open("outputs/model_results.txt", "w") as file:

    file.write("REINFORCEMENT LEARNING FOR GAMES\n")
    file.write("TIC-TAC-TOE AI USING Q-LEARNING\n\n")

    file.write("Author: Shaik Rehaman\n")
    file.write(f"Training Episodes: {EPISODES}\n")
    file.write(f"Evaluation Games: {EVALUATION_GAMES}\n")
    file.write(f"Wins: {wins}\n")
    file.write(f"Draws: {draws}\n")
    file.write(f"Losses: {losses}\n")
    file.write(f"Win Rate: {win_rate:.4f}\n")

print("\nModel and results saved successfully!")

# 12. SAMPLE GAME

print("\nSAMPLE GAME")
print("X = Trained AI | O = Random Opponent")

board = [0] * 9

while True:

    action = agent.choose_action(board, training=False)
    board[action] = 1

    print("\nAI MOVE:")
    display_board(board)

    if winner(board) == 1:
        print("AI WINS!")
        break

    if is_draw(board):
        print("GAME DRAW!")
        break

    opponent_action = random.choice(legal_actions(board))
    board[opponent_action] = -1

    print("\nOPPONENT MOVE:")
    display_board(board)

    if winner(board) == -1:
        print("OPPONENT WINS!")
        break

    if is_draw(board):
        print("GAME DRAW!")
        break

# 13. PROJECT SUMMARY

print("\nPROJECT COMPLETED")
print("Algorithm: Q-Learning")
print("Game: Tic-Tac-Toe")
print("Model: models/q_learning_model.pkl")
print("Results: outputs/")

In [ ]:
import shutil
from google.colab import files

# Create one ZIP containing models and outputs
shutil.make_archive(
    "Task_3_Results",
    "zip",
    root_dir="/content",
    base_dir="models"
)

# Add outputs to the same ZIP
import zipfile
with zipfile.ZipFile("Task_3_Results.zip", "a", zipfile.ZIP_DEFLATED) as zipf:
    for folder, _, filenames in os.walk("/content/outputs"):
        for filename in filenames:
            filepath = os.path.join(folder, filename)
            zipf.write(filepath, os.path.relpath(filepath, "/content"))

# Download ZIP to your device
files.download("Task_3_Results.zip")

In [ ]:
import os
from google.colab import files

# Create README.md
readme = """# Reinforcement Learning for Games
## Tic-Tac-Toe AI using Q-Learning

**Author:** Shaik Rehaman

### Project Description
This project develops a Tic-Tac-Toe AI agent using Reinforcement Learning.
The agent learns to choose moves through repeated games against a random opponent.

### Technologies Used
- Python
- NumPy
- Pandas
- Matplotlib
- Joblib
- Q-Learning

### Algorithm
The project uses tabular Q-Learning with an epsilon-greedy policy.

### Training
The agent is trained for 30,000 episodes and evaluated over 3,000 games.

### Results
The notebook displays the AI's wins, draws, losses, and training performance graphs.

### How to Run
1. Open the notebook in Google Colab.
2. Run all cells.
3. Check the models and outputs folders for generated files.

### Note
This is an educational project. The AI is evaluated against a random opponent
and is not guaranteed to play optimally.
"""

with open("README.md", "w") as f:
    f.write(readme)

# Create requirements.txt
requirements = """numpy
pandas
matplotlib
joblib
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)

# Package both files
import zipfile

with zipfile.ZipFile("Task_3_Documentation.zip", "w") as z:
    z.write("README.md")
    z.write("requirements.txt")

# Download
files.download("Task_3_Documentation.zip")

In [ ]:
from google.colab import files

files.download("Reinforcement_Learning_Game.ipynb")

In [ ]:
from google.colab import files

files.download("Reinforcement_Learning_Game.ipynb")

In [ ]:
from google.colab import files

files.download("Reinforcement_Learning_Game.ipynb")